In [0]:
container_path = r'abfss://sales-store@azdlssalesstore.dfs.core.windows.net/'

In [0]:
file_presencial_compras = container_path + 'landing/compras/Presencial.csv' 
file_compras_online = container_path + 'landing/compras/Online.json'

In [0]:
from pyspark.sql.functions import col, sum, lit, current_timestamp

### Compras Presencial - CSV

In [0]:
compras_presencial = (
                        spark.read.format("csv")
                                    .option("header", True)
                                    .option("sep", ";")
                                    .option("inferSchema", "false")
                                    .load(file_presencial_compras)
)

In [0]:
import re
nuevo_nombre = []

for cols in compras_presencial.columns:
    nuevo_nombre.append(
        re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', cols).lower()
    )
compras_presencial = compras_presencial.toDF(*nuevo_nombre)

In [0]:
compras_presencial = (
    compras_presencial.withColumn("tipo_compra", lit("Presencial"))
                        .withColumn("fecha_carga", current_timestamp())
                                    )

### Compras Online - JSON

In [0]:
compras_online = (
    spark.read.format("json")
              .option("multiline", True)
              .option("inferSchema", False)
              .load(file_compras_online)                  
)

In [0]:
nuevo_nombre = []

for cols in compras_online.columns:
    nuevo_nombre.append(
        re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', cols).lower()
    )
compras_online = compras_online.toDF(*nuevo_nombre)

In [0]:
compras_online = (
    compras_online.withColumn("tipo_compra", lit("Online"))
                  .withColumn("fecha_carga", current_timestamp())
)

## Unión de los dos DF

In [0]:
df_compras = compras_presencial.unionByName(compras_online, allowMissingColumns=True)

## Importación Masiva de archivos CSV

In [0]:
file_archivos_detalle = r'abfss://sales-store@azdlssalesstore.dfs.core.windows.net/'
archivos_detalle = file_archivos_detalle + 'landing/detalles/*.csv' 

In [0]:
df_detalles = (
            spark.read.format("csv")
                    .option("header", True)
                    .option("inferSchema", "false")
                    .option("sep", "|")
                    .option("mergeSchema", "true")
                    .load(archivos_detalle)
                    )

In [0]:
import re
nuevo_nombre = []

for cols in df_detalles.columns:
    nuevo_nombre.append(
        re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', cols).lower()
    )
df_detalles = df_detalles.toDF(*nuevo_nombre)

In [0]:
from pyspark.sql.functions import expr, split

#df_detalles.withColumn("tipo_archivo", expr("substring_index(_metadata.file_name, '.', 1)")).display()
#df_detalles = (df_detalles.withColumn("nombre_archivo", split(df_detalles["_metadata.file_name"], "\.csv")[0])
df_detalles = (df_detalles.withColumn("nombre_archivo", df_detalles["_metadata.file_name"])
            .withColumn("fecha_carga", current_timestamp()))

#df_detalles.withColumn("tipo_archivo", split(df_detalles["_metadata.file_name"], "\.")[0])

## Realizar poblado a las tablas DELTA 

In [0]:
df_detalles.write.mode("overwrite").format("delta").saveAsTable("linio.bronze_detalles")
df_compras.write.mode("overwrite").format("delta").saveAsTable("linio.bronze_compras")